### Fera formidável 4.6 - E se meus dados forem imagens?

#### Enunciado

Objetivo: implementar uma rede neural convolucional (CNN) utilizando PyTorch ou
lightning. Treine esta rede neural em um conjunto de dados de imagens. Explique para
o leitor como funciona a camada de convolução de uma CNN e o motivo de utilizarmos
este tipo de arquitetura quando estudamos imagens.

Dica: um dos mais famosos conjuntos de dados de imagens é o MNIST.

### Redes neurais convolucionais

#### **Camada convolucional** `nn.Conv2d`

Uma camada convolucional consiste em alguns componentes importantes, são eles:

**Kernels/filtros** `kernel_size`: São pequenas matrizes compostas por pesos em cada um de seus elementos. Esses filtros deslizam sobre a imagem realizando operações. 

**Passo** `stride`: É o tamanho do passo que o filtros usa para se mover para a próxima fração da matriz original. (no código `stride` define o tamanho do passo) 

**Preenchimento** `padding`: É a adição de pixels extras ao redor da borda da imagem de entrada, cada um desses pixels é definido como 0 e garante que o filtros se ajuste à imagem corretamente. 

No código feito temos, também, os parâmetros `in_channels` e `out_channels` que controlam, respectivamente, a quantidade de canais da imagem e a quantidade de kernels/filtros que serão utilizados. Como as imagens do MNIST são em escala de cinza o valor para o `in_channels` na primeira camada de convolução é 1.

**Mapa de recursos**: É a saída de uma camada convolucional.

Os filtros deslizam sobre a imagem de entrada multiplicando cada elemento de uma fração da matriz original pelo seu correspondente no filtros. Após isso, todos os elementos da matriz resultado desssa multiplicação são somados e resultam em um único elemento que vai compor o mapa de recursos. Como mostrado na imagem abaixo:

<img title="Camada Convolucional" alt="Alt text" src="Camada_convolucional.png">

#### **Mapa de recursos**

O mapa de recursos, que é a saída da camada convolucional, é feito a partir de diveros filtros diferentes que capturam características distintas da imagem.

#### **Camada de agrupamento**  

Consiste em uma maneira de reduzir as dimensões espaciais dos mapas de características, deixando a rede mais eficiente e reduzindo o sobreajuste. Existe dois tipos de camada de agrupamento:

**Agrupamento máximo** `nn.MaxPool2d`: Consite em pegar o maior valor de cada Kernel do mapa de recursos. 

**Agrupamento médio** `AvgPool2d`: Consiste em pegar a média dos valores de cada kernel do mapa de recursos.

A imagem abaixo mostra o funcionamento de uma camada de agrupamento:

<img title="Camada Convolucional" alt="Alt text" src="Camada_recursos.png">

#### **Modelo, perda e otimizador**

Após saber como funciona uma rede neural convolucional podemos criar a nossa rede neural pela função `CNN()`. Além disso, definimos a função de perda como a perda Cross Entropy `nn.CrossEntropyLoss()` e definimos o nosso otimizador como o otimizador Adam `optim.Adam()` que recebe os parâmetros da nossa rede neural e a nossa taxa de aprendizado. 

#### **Treinamento e acurácia do modelo**

Por fim, treinamos o modelo e podemos conferir a acurácia dele utilizando os dados de teste.

### Importando bibliotecas

In [1]:
import torch
import torch.nn as nn 
import torch.optim as optim 
import torch.nn.functional as F 
from torch.utils.data import DataLoader 
import torchvision.datasets as datasets  
import torchvision.transforms as transforms
import time

### Definindo os dados de treino e teste do banco de dados MNIST

O modulo torch possui alguns datasets famosos de fácil acesso. Por isso, o código abaixo define os nossos dados de treino e de teste de uma maneira simples.

In [2]:
df_treino = datasets.MNIST(root= 'dataset/' , train= True , transform=transforms.ToTensor(), download= True) 
df_teste = datasets.MNIST(root= 'dataset/' , train= False , transform=transforms.ToTensor(), download= True) 

### Definindo a CNN

In [3]:
class CNN(nn.Module):
    def __init__(self): 
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
        self.conv2 = nn.Conv2d(in_channels=8, out_channels=16, kernel_size=3, stride=1, padding= 1) 
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2) 
        self.fc1 = nn.Linear(16*7*7, 10) 
        
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = self.pool(x)
        x = F.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(-1, 16*7*7)
        x = F.relu(self.fc1(x))
        return x

### Carregando os dados

In [4]:
BATCH_SIZE = 32 # Número de amostras de imagens processadas ao mesmo tempo a cada iteração

loader_treino = DataLoader(dataset=df_treino, batch_size=BATCH_SIZE, shuffle= True) 
loader_teste = DataLoader(dataset=df_teste, batch_size=BATCH_SIZE, shuffle= True)

### Variáveis, modelo, perda e otimizador

A cada época a rede neural terá passado por todas as imagens do dataset.

A função Cross Entropy mede a dissimilaridade entra a distribuição de rótulos verdadeira e as previsões do modelo.

In [5]:
TAXA_APRENDIZADO = 0.001
NUM_EPOCAS = 10

In [6]:
modelo = CNN()

In [7]:
perda = nn.CrossEntropyLoss() 
otimizador = optim.Adam(modelo.parameters(), lr= TAXA_APRENDIZADO)

### Treinando a Rede Neural

In [8]:
inicio = time.time()
for epoca in range(NUM_EPOCAS):
    for images, labels in loader_treino:
        outputs = modelo(images)
        loss = perda(outputs, labels)
        otimizador.zero_grad()
        loss.backward()
        otimizador.step()
        
    print(f"Época {epoca + 1} concluída")

fim = time.time()
print()
print(f"Treinamento finalizado após {(fim - inicio) / 60} minutos")

Época 1 concluída
Época 2 concluída
Época 3 concluída
Época 4 concluída
Época 5 concluída
Época 6 concluída
Época 7 concluída
Época 8 concluída
Época 9 concluída
Época 10 concluída

Treinamento finalizado após 6.416972386837005 minutos


### Validando o modelo

`torch.argmax` pega a classe com maior probabilidade.

Note que a acurácia só foi baixa devido ao fato da baixa quantidade de épocas.

In [9]:
num_acertos = 0
num_total = 0

for imagens, rotulos in loader_teste:
    imagens = imagens
    rotulos = rotulos

    saida = modelo(imagens)                      
    previsoes = torch.argmax(saida, dim=1)       

    acertos = (previsoes == rotulos).sum().item()  
    num_acertos += acertos
    num_total += rotulos.size(0)                  

acuracia = num_acertos / num_total
print(f'Acurácia: {acuracia * 100:.2f}%')

Acurácia: 47.44%


### Conclusão 

O modulo torch é de grande utilidade para treinar redes neurais, especialmente as mais complexas como é o caso de uma Rede Neural Convolucional, , além disso, é fácil utilizar ele para treinar uma rede neural. 

### Referências

1. Cassar, Daniel R. Material didático redes neurais do ano de 2025.

BHARGAVI POYEKAR. Building simple Neural Networks using Pytorch (NN, CNN) for MNIST dataset. Disponível em: <https://medium.com/%40bpoyeka1/building-simple-neural-networks-nn-cnn-using-pytorch-for-mnist-dataset-31e459d17788>. Acesso em: 2 maio. 2025.